In [1]:
import UNet.SinkTools2 as st
import time
start_time = time.time()

/Users/nirgut/PycharmProjects/Sinkholes_ml/UNet/SinkTools2.py:16: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


# Insert Raster Path & Output Directory, Define Tile Size & Overlap

In [2]:
# input_DSM_path = '/Users/nirgut/Documents/Master_Files/pipeline_testing/DSM_crop_-32767.tif'
input_DSM_path = '/Users/nirgut/Documents/Master_Files/pipeline_testing/DSM_new_boundaries.tif'
output_DSM_tiles_dir = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/DSM_tiles/'
output_notempty_DSM_tiles_dir = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/DSM_tiles/not_empty'
output_empty_DSM_tiles_dir = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/DSM_tiles/empty'
# input_mask_path = '/Users/nirgut/Documents/Master_Files/pipeline_testing/Mask_crop_-32767.tif'
input_mask_path = '/Users/nirgut/Documents/Master_Files/pipeline_testing/Mask_new_boundaries.tif'
output_mask_tiles_dir = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/Mask_tiles/'
output_notempty_mask_tiles_dir = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/Mask_tiles/not_empty'

DSM_norm_empty = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/DSM_tiles/normalized/empty'
DSM_norm_notempty = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/DSM_tiles/normalized/not_empty'

# Define the size of the tiles
tile_size_x = 256  # width of each tile in pixels
tile_size_y = 256  # height of each tile in pixels
overlap_x = 50  # Example overlap in x direction
overlap_y = 50  # Example overlap in y direction

In [3]:
# creating tiles
st.divide_raster_to_tiles(input_DSM_path, output_DSM_tiles_dir, tile_size_x, tile_size_y, overlap_x, overlap_y)
st.divide_raster_to_tiles(input_mask_path, output_mask_tiles_dir, tile_size_x, tile_size_y, overlap_x, overlap_y)

  0%|          | 0/168 [00:04<?, ?it/s]


KeyboardInterrupt: 

# Filtering tiles by mask

In [ ]:
st.check_empty_rasters(output_mask_tiles_dir, 10, output_DSM_tiles_dir)

In [ ]:
import os
import numpy as np
from osgeo import gdal, gdalconst

def normalize_raster(input_raster_path, output_raster_path):
    """
    Normalize the values of a raster to the range 0-1 and save it as a new raster.

    :param input_raster_path: Path to the input raster file.
    :param output_raster_path: Path to the output raster file.
    """
    # Open the input raster
    dataset = gdal.Open(input_raster_path, gdalconst.GA_ReadOnly)
    if not dataset:
        print(f"Failed to open raster file: {input_raster_path}")
        return

    # Read the raster data
    band = dataset.GetRasterBand(1)
    array = band.ReadAsArray()

    # Normalize the array to the range 0-1
    min_val = np.min(array)
    max_val = np.max(array)
    norm_array = (array - min_val) / (max_val - min_val)

    # Create the output raster
    driver = gdal.GetDriverByName('GTiff')
    out_dataset = driver.Create(output_raster_path, dataset.RasterXSize, dataset.RasterYSize, 1, gdal.GDT_Float32)
    if not out_dataset:
        print(f"Failed to create output raster file: {output_raster_path}")
        return
    
    out_dataset.SetGeoTransform(dataset.GetGeoTransform())
    out_dataset.SetProjection(dataset.GetProjection())

    # Write the normalized data to the output raster
    out_band = out_dataset.GetRasterBand(1)
    out_band.WriteArray(norm_array)
    out_band.SetNoDataValue(-32767)

    # Clean up
    band = None
    dataset = None
    out_band = None
    out_dataset = None

def normalize_rasters_in_folder(input_folder, output_folder):
    """
    Normalize the values of all rasters in a folder to the range 0-1 and save them as new rasters.

    :param input_folder: Path to the folder containing input rasters.
    :param output_folder: Path to the folder where output rasters will be saved.
    """
    # Check if the output folder exists, create it if not
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Process each raster file in the input folder
    for filename in os.listdir(input_folder):
        if filename.endswith('.tif'):
            input_raster_path = os.path.join(input_folder, filename)
            output_raster_path = os.path.join(output_folder, filename)
            print(f"Normalizing {input_raster_path} and saving to {output_raster_path}")
            normalize_raster(input_raster_path, output_raster_path)





In [ ]:
normalize_rasters_in_folder(output_notempty_DSM_tiles_dir, DSM_norm_notempty)
normalize_rasters_in_folder(output_empty_DSM_tiles_dir, DSM_norm_empty)

In [ ]:
st.split_data_and_masks(DSM_norm_notempty, output_notempty_mask_tiles_dir, 0.8)

In [ ]:
end_time = time.time()
print("Total time: ", end_time - start_time)
